# Loading and Cleaning Data

## Setup

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

# The kernel may start in notebooks/ or the project root, so find the root by requirements.txt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW = ROOT / "data" / "raw"
ROOT

PosixPath('/home/souparno/Projects/Indian-Trade-Basket_What-We-Buy-and-What-We-Sell')

## The files and the manifest

In [2]:
files = sorted(RAW.glob("*/*.xlsx"))
manifest = pd.read_csv(RAW / "manifest.csv")
print(len(files), "files")
manifest[["labelled", "flow", "years", "rows", "total_y1", "total_y2", "gap_y1", "gap_y2"]]

16 files


,labelled,flow,years,rows,total_y1,total_y2,gap_y1,gap_y2
0,tradestat_import_commodity-wise_FY2022-23_FY20...,import,"['2022-2023', '2023-2024']",10769,715968.90,673991.95,0.54,0.40
1,tradestat_import_commodity-wise_FY2023-24_FY20...,import,"['2023-2024', '2024-2025']",10681,673991.95,721200.22,0.40,0.54
2,tradestat_import_commodity-wise_FY2024-25_FY20...,import,"['2024-2025', '2025-2026']",10850,721200.22,775713.17,0.54,1.02
3,tradestat_import_commodity-wise_FY2017-18_FY20...,import,"['2017-2018', '2018-2019']",10264,465580.99,514078.42,0.49,0.14
4,tradestat_import_commodity-wise_FY2018-19_FY20...,import,"['2018-2019', '2019-2020']",10379,514078.42,474709.28,0.14,1.05
5,tradestat_import_commodity-wise_FY2019-20_FY20...,import,"['2019-2020', '2020-2021']",10555,474709.28,394435.88,1.05,0.55
6,tradestat_import_commodity-wise_FY2020-21_FY20...,import,"['2020-2021', '2021-2022']",10344,394435.88,613052.05,0.55,0.47
7,tradestat_import_commodity-wise_FY2021-22_FY20...,import,"['2021-2022', '2022-2023']",10740,613052.05,715968.90,0.47,0.54
8,tradestat_export_commodity-wise_FY2017-18_FY20...,export,"['2017-2018', '2018-2019']",10942,303526.16,330078.09,0.65,1.42
9,tradestat_export_commodity-wise_FY2018-19_FY20...,export,"['2018-2019', '2019-2020']",10952,330078.09,313361.04,1.42,0.73


## One report exactly as stored, top and bottom

In [3]:
path = RAW / "export" / "tradestat_export_commodity-wise_FY2022-23_FY2023-24.xlsx"
raw = pd.read_excel(path, header=None)
display(raw.head(5))
display(raw.tail(3))

,0,1,2,3,4,5,6,7
0,TradeStat->Eidb->Export->Commodity-wise,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Report Generated on: 10/26/2026 - Values in US...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S.No.,HSCode,Commodity,2022 - 2023,%Share,2023 - 2024,%Share,%Growth
3,1,01012100,PURE-BRED BREEDING HORSES,0.21,0,0,0,NaN
4,2,01012990,OTHER HORSES,0.09,0,0.02,0,-78.91


,0,1,2,3,4,5,6,7
11536,11534,99999998,UNSPECIFIED ITEMS,0.03,0,0,0,NaN
11537,11535,99999999,SUPPRESSED COMMODITY,103.73,0.023,186.99,0.0428,80.27
11538,India's Total Export,India's Total Export,India's Total Export,451070,NaN,437072.03,NaN,-3.1


## Read with the real header row

In [4]:
df = pd.read_excel(path, header=2)
print(df.columns.tolist())
df.dtypes

['S.No.', 'HSCode', 'Commodity', '2022 - 2023', '%Share', '2023 - 2024', '%Share.1', '%Growth']


S.No.           object
HSCode          object
Commodity          str
2022 - 2023    float64
%Share         float64
2023 - 2024    float64
%Share.1       float64
%Growth        float64
dtype: object

## Storing of HS codes

In [5]:
codes = df["HSCode"].iloc[:-1]                      # drop the India's Total row
is_text = codes.map(lambda x: isinstance(x, str))
print("text codes:   ", is_text.sum(), "| chapters:", sorted(codes[is_text].str[:2].unique()))
print("numeric codes:", (~is_text).sum(), "| smallest:", codes[~is_text].min())

text codes:    714 | chapters: ['01', '02', '03', '04', '05', '06', '07', '08', '09']
numeric codes: 10821 | smallest: 10011100


## Prototype of read_report

In [6]:
import re

YEAR = re.compile(r"(\d{4})\s*-\s*(\d{4})")

def fy_label(text):
    """'2022 - 2023' -> '2022-23'"""
    m = YEAR.fullmatch(str(text).strip())
    if m is None:
        raise ValueError(f"not a fiscal-year header: {text!r}")
    a, b = m.groups()
    return f"{a}-{b[2:]}"

def read_report(path):
    """Parse one TradeStat commodity-wise report into (meta, rows)."""
    path = Path(path)
    raw = pd.read_excel(path, header=None,  converters={1: str})   # column 1 = HSCode, kept as text
    title, subtitle, header = str(raw.iat[0, 0]), str(raw.iat[1, 0]), raw.iloc[2]

    # Title: TradeStat->Eidb->Export->Commodity-wise
    flow = title.split("->")[2].strip().lower()
    assert "US $ Million" in subtitle, subtitle
    fy1, fy2 = fy_label(header[3]), fy_label(header[5])

    body = raw.iloc[3:, [1, 2, 3, 5]]                        # HSCode, Commodity, year-1, year-2
    body.columns = ["hs8", "description", "y1", "y2"]
    is_total = body["hs8"].str.startswith("India's Total")
    assert is_total.sum() == 1 and is_total.iloc[-1], "expected one total row, at the bottom"

    total = body[is_total].iloc[0]
    rows = body[~is_total].reset_index(drop=True)
    rows["hs8"] = rows["hs8"].str.strip().str.zfill(8)
    rows[["y1", "y2"]] = rows[["y1", "y2"]].astype("float64")

    m = re.search(r"Report Generated on:\s*([\d/]+)", subtitle)
    meta = dict(file=path.name, flow=flow, fy1=fy1, fy2=fy2,
                total1=float(total["y1"]), total2=float(total["y2"]),
                generated=m.group(1) if m else None)
    return meta, rows

meta, rows = read_report(path)
display(meta)
display(rows.head())
display(rows.tail(3))
rows.dtypes

{'file': 'tradestat_export_commodity-wise_FY2022-23_FY2023-24.xlsx',
 'flow': 'export',
 'fy1': '2022-23',
 'fy2': '2023-24',
 'total1': 451070.0,
 'total2': 437072.03,
 'generated': '10/26/2026'}

,hs8,description,y1,y2
0,01012100,PURE-BRED BREEDING HORSES,0.21,0.00
1,01012990,OTHER HORSES,0.09,0.02
2,01023100,PURE-BRED BREEDING BUFFALOES,0.06,0.44
3,01029010,LIVE BOVINE ANMLS OTHR CATTLE AND BUFFALO PUR...,0.00,0.06
4,01041090,"OTHERS,EXCLDNG LIVE SHEEP AND LAMB FOR BR...",0.05,2.79


,hs8,description,y1,y2
11532,99930090,SPECIAL TRANSACTIONS N.E.S.,0.10,0.55
11533,99999998,UNSPECIFIED ITEMS,0.03,0.00
11534,99999999,SUPPRESSED COMMODITY,103.73,186.99


hs8                str
description        str
y1             float64
y2             float64
dtype: object

## Read all 16 and check

In [7]:
reports = [read_report(p) for p in files]

check = pd.DataFrame([dict(m, n_rows=len(r), sum1=r["y1"].sum(), sum2=r["y2"].sum()) for m, r in reports])
check["ratio1"] = (check["sum1"] / check["total1"]).round(4)
check["ratio2"] = (check["sum2"] / check["total2"]).round(4)

rows_match = (check.set_index("file")["n_rows"] == manifest.set_index("labelled").loc[check["file"], "rows"].values).all()
codes_ok = all(r["hs8"].str.fullmatch(r"\d{8}").all() for _, r in reports)
no_blanks = sum(r[["y1", "y2"]].isna().sum().sum() for _, r in reports) == 0
print("row counts match manifest:", rows_match, "| all codes 8 digits:", codes_ok, "| no blank values:", no_blanks)
check[["file", "flow", "fy1", "fy2", "n_rows", "total1", "total2", "ratio1", "ratio2"]]


row counts match manifest: True | all codes 8 digits: True | no blank values: True


,file,flow,fy1,fy2,n_rows,total1,total2,ratio1,ratio2
0,tradestat_export_commodity-wise_FY2017-18_FY20...,export,2017-18,2018-19,10942,303526.16,330078.09,1.0,1.0
1,tradestat_export_commodity-wise_FY2018-19_FY20...,export,2018-19,2019-20,10952,330078.09,313361.04,1.0,1.0
2,tradestat_export_commodity-wise_FY2019-20_FY20...,export,2019-20,2020-21,11198,313361.04,291808.48,1.0,1.0
3,tradestat_export_commodity-wise_FY2020-21_FY20...,export,2020-21,2021-22,11127,291808.48,422004.40,1.0,1.0
4,tradestat_export_commodity-wise_FY2021-22_FY20...,export,2021-22,2022-23,11538,422004.40,451070.00,1.0,1.0
5,tradestat_export_commodity-wise_FY2022-23_FY20...,export,2022-23,2023-24,11535,451070.00,437072.03,1.0,2.0
6,tradestat_export_commodity-wise_FY2023-24_FY20...,export,2023-24,2024-25,11434,437072.03,437704.58,2.0,1.0
7,tradestat_export_commodity-wise_FY2024-25_FY20...,export,2024-25,2025-26,11562,437704.58,441452.14,1.0,1.0
8,tradestat_import_commodity-wise_FY2017-18_FY20...,import,2017-18,2018-19,10264,465580.99,514078.42,1.0,1.0
9,tradestat_import_commodity-wise_FY2018-19_FY20...,import,2018-19,2019-20,10379,514078.42,474709.28,1.0,1.0
